In [2]:
!pip install ultralytics > /dev/null || pip uninstall -y albumentations > /dev/null || pip install albumentationsx > /dev/null || echo -e "INSTALLATION COMPLETED"

In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
!pwd

CUDA available: True
CUDA device: Tesla P100-PCIE-16GB
/kaggle/working


In [4]:
%%writefile data.yaml
train:
- /kaggle/input/aod-4-yolo/train/images
val:
- /kaggle/input/aod-4-yolo/valid/images
test:
- /kaggle/input/aod-4-yolo/test/images

names:
    0: airplane
    1: bird
    2: drone
    3: helicopter

Writing data.yaml


In [5]:
%%writefile custom_yolov11n.yaml
# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

# Ultralytics YOLO11 object detection model with P3/8 - P5/32 outputs
# Model docs: https://docs.ultralytics.com/models/yolo11
# Task docs: https://docs.ultralytics.com/tasks/detect

# Parameters

nc: 4 # [airplane, bird, drone, helicopter]
# CHANGES: nc only

scales: # model compound scaling constants, i.e. 'model=yolo11n.yaml' will call yolo11.yaml with scale 'n'
  # [depth, width, max_channels]
  n: [0.50, 0.25, 1024] # summary: 181 layers, 2624080 parameters, 2624064 gradients, 6.6 GFLOPs
  s: [0.50, 0.50, 1024] # summary: 181 layers, 9458752 parameters, 9458736 gradients, 21.7 GFLOPs
  m: [0.50, 1.00, 512] # summary: 231 layers, 20114688 parameters, 20114672 gradients, 68.5 GFLOPs
  l: [1.00, 1.00, 512] # summary: 357 layers, 25372160 parameters, 25372144 gradients, 87.6 GFLOPs
  x: [1.00, 1.50, 512] # summary: 357 layers, 56966176 parameters, 56966160 gradients, 196.0 GFLOPs

# YOLO11n backbone
backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [64, 3, 2]] # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]] # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]] # 3-P3/8
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]] # 5-P4/16
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]] # 7-P5/32
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]] # 9
  - [-1, 2, C2PSA, [1024]] # 10

# YOLO11n head
head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]] # cat backbone P4
  - [-1, 2, C3k2, [512, False]] # 13

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]] # cat backbone P3
  - [-1, 2, C3k2, [256, False]] # 16 (P3/8-small)

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]] # cat head P4
  - [-1, 2, C3k2, [512, False]] # 19 (P4/16-medium)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]] # cat head P5
  - [-1, 2, C3k2, [1024, True]] # 22 (P5/32-large)

  - [[16, 19, 22], 1, Detect, [nc]] # Detect(P3, P4, P5)

Writing custom_yolov11n.yaml


In [11]:
# train_loop.py
import argparse
import glob
import os
from dataclasses import dataclass
from types import SimpleNamespace

import pandas as pd
import torch
from torch import Tensor
from torch.nn.utils.rnn import pad_sequence
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset
from torchvision.io import decode_image
from tqdm.auto import tqdm

from ultralytics import YOLO
from ultralytics.utils.loss import v8DetectionLoss


# -----------------------------
# Utils
# -----------------------------
def get_device() -> torch.device:
    # ✅ Force cuda:0 on Kaggle when available
    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


# -----------------------------
# Dataset
# -----------------------------
class AodDataset(Dataset):
    """
    Read images + YOLO labels (cls xc yc w h) normalized [0..1].

    Return:
      - image: uint8 tensor [C,H,W] on CPU
      - targets: float tensor [N,5] on CPU (or [[-1]*5] if empty/missing).
    """

    def __init__(
        self,
        images_dir: str,
        labels_dir: str,
        images_ext: str = "jpg",
        labels_ext: str = "txt",
        transforms=None,
        verbose_bad_labels: bool = False,
    ):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.images_ext = images_ext
        self.labels_ext = labels_ext
        self.transforms = transforms
        self.verbose_bad_labels = verbose_bad_labels

        img_glob = os.path.join(images_dir, f"*.{images_ext}")
        self.images = sorted(glob.glob(img_glob))
        if len(self.images) == 0:
            raise FileNotFoundError(f"No images found: {img_glob}")

        self.miss_labeled: list[str] = []

    def _labels_path_from_image(self, image_path: str) -> str:
        base = os.path.basename(image_path)
        stem = base[: -(len(self.images_ext) + 1)]
        return os.path.join(self.labels_dir, f"{stem}.{self.labels_ext}")

    def _read_labels(self, labels_path: str) -> Tensor:
        if not os.path.exists(labels_path):
            self.miss_labeled.append(labels_path)
            return torch.tensor([[-1, -1, -1, -1, -1]], dtype=torch.float32)

        labels: list[list[float]] = []
        with open(labels_path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) != 5:
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: invalid values count: {line}")
                    continue
                try:
                    c, xc, yc, w, h = map(float, parts)
                except ValueError:
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: cannot parse: {line}")
                    continue

                if not (0.0 <= xc <= 1.0 and 0.0 <= yc <= 1.0 and 0.0 < w <= 1.0 and 0.0 < h <= 1.0):
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: invalid bbox: {line}")
                    continue

                labels.append([c, xc, yc, w, h])

        if len(labels) == 0:
            self.miss_labeled.append(labels_path)
            return torch.tensor([[-1, -1, -1, -1, -1]], dtype=torch.float32)

        return torch.tensor(labels, dtype=torch.float32)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        image_path = self.images[idx]
        img = decode_image(image_path)  # uint8 [C,H,W] on CPU
        if self.transforms is not None:
            img = self.transforms(img)

        labels_path = self._labels_path_from_image(image_path)
        targets = self._read_labels(labels_path)
        return img, targets

    def __len__(self) -> int:
        return len(self.images)


# -----------------------------
# Collate & Batch builder
# -----------------------------
def collate_fn(batch: list[tuple[Tensor, Tensor]]) -> tuple[Tensor, Tensor]:
    images, targets = zip(*batch)
    images = torch.stack(images, dim=0)
    padded_targets = pad_sequence(list(targets), batch_first=True, padding_value=-1.0)
    return images, padded_targets


def prepare_batch_dict(images: Tensor, targets: Tensor) -> dict:
    valid_mask = targets[..., 0] != -1
    if not valid_mask.any():
        return {
            "img": images,
            "batch_idx": torch.empty((0,), device=images.device, dtype=torch.int64),
            "cls": torch.empty((0,), device=images.device, dtype=torch.float32),
            "bboxes": torch.empty((0, 4), device=images.device, dtype=torch.float32),
        }

    valid_targets = targets[valid_mask]  # [M,5]

    per_img_counts = valid_mask.sum(dim=1).to(torch.int64)
    batch_idx_list = []
    for i, cnt in enumerate(per_img_counts.tolist()):
        if cnt > 0:
            batch_idx_list.append(torch.full((cnt,), i, device=images.device, dtype=torch.int64))
    batch_idx = (
        torch.cat(batch_idx_list, dim=0)
        if batch_idx_list
        else torch.empty((0,), device=images.device, dtype=torch.int64)
    )

    cls = valid_targets[:, 0].to(torch.float32)
    bboxes = valid_targets[:, 1:].to(torch.float32)
    return {"img": images, "batch_idx": batch_idx, "cls": cls, "bboxes": bboxes}


# -----------------------------
# Train / Valid
# -----------------------------
def train_one_epoch(
    model: YOLO,
    optimizer: Adam,
    dataloader: DataLoader,
    device: torch.device,
    epoch: int,
    max_train_steps: int = 0,
    debug_print_first_batch: bool = True,
) -> pd.DataFrame:
    model.model.train(True)
    torch.set_grad_enabled(True)
    loss_fn = v8DetectionLoss(model.model)

    rows = []
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"train epoch {epoch}")

    for step, (images, targets) in pbar:
        if max_train_steps > 0 and step >= max_train_steps:
            break

        images = images.to(device, non_blocking=True).float() / 255.0
        targets = targets.to(device, non_blocking=True)

        if debug_print_first_batch and step == 0:
            print("[DEBUG] images device:", images.device)
            print("[DEBUG] targets device:", targets.device)
            print("[DEBUG] model device:", next(model.model.parameters()).device)

        batch = prepare_batch_dict(images, targets)
        if batch["bboxes"].numel() == 0:
            pbar.set_postfix_str("skip(empty)")
            continue

        optimizer.zero_grad(set_to_none=True)

        # ✅ Prevent inference_mode leakage from Ultralytics internals in notebooks
        with torch.inference_mode(False):
            with torch.enable_grad():
                preds = model.model(images)
                batch_loss, last_loss = loss_fn(preds, batch)
                loss_scalar = batch_loss.sum()

        if not torch.isfinite(loss_scalar):
            pbar.set_postfix_str("skip(non-finite)")
            continue

        if not loss_scalar.requires_grad:
            raise RuntimeError(
                f"loss_scalar.requires_grad=False at step {step}. "
                f"Grad is disabled globally or inference_mode is active."
            )

        loss_scalar.backward()
        optimizer.step()

        box_loss, cls_loss, dfl_loss = last_loss.detach().cpu().tolist()
        row = {
            "epoch": epoch,
            "step": step,
            "box": float(box_loss),
            "cls": float(cls_loss),
            "dfl": float(dfl_loss),
        }
        rows.append(row)
        pbar.set_postfix({"box": f"{row['box']:.4f}", "cls": f"{row['cls']:.4f}", "dfl": f"{row['dfl']:.4f}"})

    return pd.DataFrame(rows)


@torch.no_grad()
def valid_loss_only(
    model: YOLO,
    dataloader: DataLoader,
    device: torch.device,
    epoch: int,
    max_val_steps: int = 0,
) -> pd.DataFrame:
    model.model.eval()
    loss_fn = v8DetectionLoss(model.model)

    rows = []
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"valid(loss) epoch {epoch}")

    for step, (images, targets) in pbar:
        if max_val_steps > 0 and step >= max_val_steps:
            break

        images = images.to(device, non_blocking=True).float() / 255.0
        targets = targets.to(device, non_blocking=True)

        batch = prepare_batch_dict(images, targets)
        if batch["bboxes"].numel() == 0:
            continue

        preds = model.model(images)
        _, last_loss = loss_fn(preds, batch)

        box_loss, cls_loss, dfl_loss = last_loss.detach().cpu().tolist()
        rows.append(
            {"epoch": epoch, "batch": step, "box": float(box_loss), "cls": float(cls_loss), "dfl": float(dfl_loss)}
        )

    return pd.DataFrame(rows)


def yolo_val_metrics(model: YOLO, data_yaml: str, device_id=0, workers: int = 0):
    # plots=False để tránh warning matplotlib & nhẹ hơn
    return model.val(data=data_yaml, device=device_id, workers=workers, verbose=False, plots=False)


# -----------------------------
# Main
# -----------------------------
@dataclass
class Config:
    base_path: str
    data_yaml: str
    epochs: int = 2
    batch_size: int = 4
    lr: float = 3e-4
    num_workers: int = 2
    img_ext: str = "jpg"
    label_ext: str = "txt"
    checkpoint_dir: str = "checkpoints"

    # Debug helpers (0 means "no limit")
    max_train_steps: int = 0
    max_val_steps: int = 0


def main(cfg: Config) -> None:
    device = get_device()
    print(f"[INFO] device = {device}")
    if device.type == "cuda":
        print("[INFO] CUDA available:", torch.cuda.is_available())
        print("[INFO] CUDA device:", torch.cuda.get_device_name(0))

    ensure_dir(cfg.checkpoint_dir)

    model = YOLO(model="custom_yolov11n.yaml", task="detect", verbose=True)
    model.model.args = SimpleNamespace(box=15.0, cls=0.5, dfl=2.25)  # type: ignore
    model.model.to(device)  # type: ignore

    print("[DEBUG] model param device:", next(model.model.parameters()).device)

    optimizer = Adam(model.model.parameters(), lr=cfg.lr)

    train_dataset = AodDataset(
        images_dir=os.path.join(cfg.base_path, "train/images"),
        labels_dir=os.path.join(cfg.base_path, "train/labels"),
        images_ext=cfg.img_ext,
        labels_ext=cfg.label_ext,
        verbose_bad_labels=False,
    )
    val_dataset = AodDataset(
        images_dir=os.path.join(cfg.base_path, "valid/images"),
        labels_dir=os.path.join(cfg.base_path, "valid/labels"),
        images_ext=cfg.img_ext,
        labels_ext=cfg.label_ext,
        verbose_bad_labels=False,
    )

    pin = torch.cuda.is_available()
    workers = cfg.num_workers

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=workers,
        pin_memory=pin,
        persistent_workers=(workers > 0),
        drop_last=False,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=workers,
        pin_memory=pin,
        persistent_workers=(workers > 0),
        drop_last=False,
    )

    train_all = []

    best_score = float("inf")
    best_epoch = -1
    best_path = os.path.join(cfg.checkpoint_dir, "best.pt")
    last_path = os.path.join(cfg.checkpoint_dir, "last.pt")

    for epoch in range(1, cfg.epochs + 1):
        print(f"\n==================== EPOCH {epoch}/{cfg.epochs} ====================")

        train_df = train_one_epoch(
            model,
            optimizer,
            train_loader,
            device,
            epoch,
            max_train_steps=cfg.max_train_steps,
            debug_print_first_batch=(epoch == 1),
        )
        train_all.append(train_df)

        if len(train_df) > 0:
            train_score = train_df[["box", "cls", "dfl"]].mean().sum()
            print(f"[EPOCH {epoch}] train_score={train_score:.6f} | best_score={best_score:.6f}")

            if train_score < best_score:
                best_score = train_score
                best_epoch = epoch
                model.save(best_path)
                print(f"[BEST] updated → epoch {best_epoch}, saved to {best_path}")
        else:
            print("[TRAIN] no valid batches produced loss (all empty labels?)")

        model.save(last_path)
        ckpt_path = os.path.join(cfg.checkpoint_dir, f"epoch_{epoch}.pt")
        model.save(ckpt_path)
        print(f"[INFO] saved checkpoint: {ckpt_path} (and last.pt)")

    train_csv = os.path.join(cfg.checkpoint_dir, "train_losses.csv")
    pd.concat(train_all, axis=0, ignore_index=True).to_csv(train_csv, index=False)
    print(f"[INFO] saved train logs: {train_csv}")

    print("\n==================== TRAIN SUMMARY ====================")
    print(f"[BEST MODEL] epoch = {best_epoch}")
    print(f"[BEST MODEL] score = {best_score:.6f}")
    print(f"[BEST MODEL] path  = {best_path}")
    print(f"[LAST MODEL] path  = {last_path}")

    print("\n==================== FINAL EVALUATION ====================")

    eval_ckpt = best_path if os.path.exists(best_path) else last_path
    print(f"[EVAL] loading checkpoint: {eval_ckpt}")

    eval_model = YOLO(eval_ckpt)
    eval_model.model.args = SimpleNamespace(box=15.0, cls=0.5, dfl=2.25)  # type: ignore
    eval_model.model.to(device)  # type: ignore

    val_loss_df = valid_loss_only(
        eval_model,
        val_loader,
        device,
        epoch=best_epoch if os.path.exists(best_path) else cfg.epochs,
        max_val_steps=cfg.max_val_steps,
    )
    val_csv = os.path.join(cfg.checkpoint_dir, "val_losses.csv")
    val_loss_df.to_csv(val_csv, index=False)
    if len(val_loss_df) > 0:
        print("[FINAL VALID-LOSS] mean:", val_loss_df[["box", "cls", "dfl"]].mean().to_dict())
    else:
        print("[FINAL VALID-LOSS] no valid batches (empty labels?)")
    print(f"[INFO] saved val losses: {val_csv}")

    try:
        print("[FINAL VALID-METRICS] running ultralytics val() ...")
        device_id = 0 if device.type == "cuda" else "cpu"
        metrics = yolo_val_metrics(eval_model, cfg.data_yaml, device_id=device_id, workers=cfg.num_workers)

        print(f"Precision: {metrics.box.mp:.4f}")
        print(f"Recall:    {metrics.box.mr:.4f}")
        print(f"mAP@0.5:   {metrics.box.map50:.4f}")
        print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
    except Exception as e:
        print(f"[WARN] final model.val() failed: {e}")


if __name__ == "__main__":

    def running_in_notebook():
        try:
            from IPython import get_ipython

            return get_ipython() is not None
        except Exception:
            return False

    if running_in_notebook():
        print("[INFO] Running in Kaggle/Colab Notebook → using hardcoded config")
        cfg = Config(
            base_path="/kaggle/input/aod-4-yolo",
            data_yaml="/kaggle/working/data.yaml",
            epochs=4,
            batch_size=4,
            lr=3e-4,
            num_workers=2,  # ✅ try 2; set 0 if you see worker issues
            max_train_steps=1000,
            max_val_steps=0,
        )
        main(cfg)
    else:
        parser = argparse.ArgumentParser()
        parser.add_argument("--base_path", type=str, required=True)
        parser.add_argument("--data_yaml", type=str, required=True)
        parser.add_argument("--epochs", type=int, default=2)
        parser.add_argument("--batch_size", type=int, default=4)
        parser.add_argument("--lr", type=float, default=3e-4)
        parser.add_argument("--num_workers", type=int, default=2)
        parser.add_argument("--img_ext", type=str, default="jpg")
        parser.add_argument("--label_ext", type=str, default="txt")
        parser.add_argument("--checkpoint_dir", type=str, default="checkpoints")
        parser.add_argument("--max_train_steps", type=int, default=0)
        parser.add_argument("--max_val_steps", type=int, default=0)

        args = parser.parse_args()
        cfg = Config(**vars(args))
        main(cfg)

[INFO] Running in Kaggle/Colab Notebook → using hardcoded config
[INFO] device = cuda:0
[INFO] CUDA available: True
[INFO] CUDA device: Tesla P100-PCIE-16GB

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultral

train epoch 1:   0%|          | 0/3941 [00:00<?, ?it/s]

[DEBUG] images device: cuda:0
[DEBUG] targets device: cuda:0
[DEBUG] model device: cuda:0
[EPOCH 1] train_score=15.400596 | best_score=inf
[BEST] updated → epoch 1, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_1.pt (and last.pt)

==================== EPOCH 2/4 ====================


train epoch 2:   0%|          | 0/3941 [00:00<?, ?it/s]

[EPOCH 2] train_score=11.379105 | best_score=15.400596
[BEST] updated → epoch 2, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_2.pt (and last.pt)

==================== EPOCH 3/4 ====================


...

[EPOCH 3] train_score=10.365357 | best_score=11.379105
[BEST] updated → epoch 3, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_3.pt (and last.pt)

==================== EPOCH 4/4 ====================


train epoch 4:   0%|          | 0/3941 [00:00<?, ?it/s]

[EPOCH 4] train_score=9.761502 | best_score=10.365357
[BEST] updated → epoch 4, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_4.pt (and last.pt)
[INFO] saved train logs: checkpoints/train_losses.csv

==================== TRAIN SUMMARY ====================
[BEST MODEL] epoch = 4
[BEST MODEL] score = 9.761502
[BEST MODEL] path  = checkpoints/best.pt
[LAST MODEL] path  = checkpoints/last.pt

==================== FINAL EVALUATION ====================
[EVAL] loading checkpoint: checkpoints/best.pt


valid(loss) epoch 4:   0%|          | 0/1129 [00:00<?, ?it/s]

[FINAL VALID-LOSS] mean: {'box': 4.05320149505085, 'cls': 2.2824154971426376, 'dfl': 3.49754746997303}
[INFO] saved val losses: checkpoints/val_losses.csv
[FINAL VALID-METRICS] running ultralytics val() ...
Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
custom_YOLOv11n summary (fused): 101 layers, 2,582,932 parameters, 13,260 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 93.6±36.5 MB/s, size: 54.4 KB)
val: Scanning /kaggle/input/aod-4-yolo/valid/labels... 4514 images, 125 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4514/4514 1.2Kit/s 3.7s0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/aod-4-yolo/valid is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 283/283 11.7it/s 24.2s<0.1s
                   all       4514       6369      0.299      0.291      0.221     0.0829
Speed: 0.8ms preprocess, 2.2ms inference, 0.0ms loss, 0.8ms

In [12]:
%%writefile hoangtn_CoordConv.py

import torch
import torch.nn as nn
from ultralytics.nn.modules.conv import Conv

class CoordConv(nn.Module):
    """
    CoordConv block (drop-in replacement for Ultralytics Conv)
    - Adds 2 coordinate channels (x, y) then applies a Conv.
    - x,y are normalized to [-1, 1] by default.
    """
    def __init__(
        self,
        c1: int,          # input channels
        c2: int,          # output channels
        k: int = 1,       # kernel
        s: int = 1,       # stride
        p=None,           # padding (None -> Ultralytics autopad inside Conv)
        g: int = 1,       # groups
        d: int = 1,       # dilation
        act=True,         # activation
        normalize: bool = True,  # normalize coords to [-1,1]
    ):
        super().__init__()
        self.normalize = normalize
        # Note: +2 channels for (x_coord, y_coord)
        self.conv = Conv(c1 + 2, c2, k, s, p, g, d, act)

    @staticmethod
    def _make_coord_grid(h: int, w: int, device, dtype, normalize: bool):
        # Create meshgrid (y, x) with shape [H,W]
        if normalize:
            xs = torch.linspace(-1.0, 1.0, w, device=device, dtype=dtype)
            ys = torch.linspace(-1.0, 1.0, h, device=device, dtype=dtype)
        else:
            xs = torch.arange(w, device=device, dtype=dtype)
            ys = torch.arange(h, device=device, dtype=dtype)

        yy, xx = torch.meshgrid(ys, xs, indexing="ij")  # [H,W], [H,W]
        # Return as [1,2,H,W]
        coord = torch.stack([xx, yy], dim=0).unsqueeze(0)
        return coord

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        coord = self._make_coord_grid(h, w, x.device, x.dtype, self.normalize)
        if coord.shape[0] != b:
            coord = coord.repeat(b, 1, 1, 1)  # [B,2,H,W]
        x = torch.cat([x, coord], dim=1)      # [B,C+2,H,W]
        return self.conv(x)

Overwriting hoangtn_CoordConv.py


In [13]:
# train_loop.py
import argparse
from dataclasses import dataclass

import pandas as pd
import torch
import torch.nn as nn

############################################################
# from hoangtn import HoangTN
from hoangtn_CoordConv import CoordConv
from torch import Tensor
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset

from ultralytics import YOLO

############################################################


# -----------------------------
# Utils
# -----------------------------
def get_device() -> torch.device:
    # ✅ Force cuda:0 on Kaggle when available
    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


# -----------------------------
# Dataset
# -----------------------------
class AodDataset(Dataset):
    """
    Read images + YOLO labels (cls xc yc w h) normalized [0..1].

    Return:
      - image: uint8 tensor [C,H,W] on CPU
      - targets: float tensor [N,5] on CPU (or [[-1]*5] if empty/missing).
    """

    def __init__(
        self,
        images_dir: str,
        labels_dir: str,
        images_ext: str = "jpg",
        labels_ext: str = "txt",
        transforms=None,
        verbose_bad_labels: bool = False,
    ):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.images_ext = images_ext
        self.labels_ext = labels_ext
        self.transforms = transforms
        self.verbose_bad_labels = verbose_bad_labels

        img_glob = os.path.join(images_dir, f"*.{images_ext}")
        self.images = sorted(glob.glob(img_glob))
        if len(self.images) == 0:
            raise FileNotFoundError(f"No images found: {img_glob}")

        self.miss_labeled: list[str] = []

    def _labels_path_from_image(self, image_path: str) -> str:
        base = os.path.basename(image_path)
        stem = base[: -(len(self.images_ext) + 1)]
        return os.path.join(self.labels_dir, f"{stem}.{self.labels_ext}")

    def _read_labels(self, labels_path: str) -> Tensor:
        if not os.path.exists(labels_path):
            self.miss_labeled.append(labels_path)
            return torch.tensor([[-1, -1, -1, -1, -1]], dtype=torch.float32)

        labels: list[list[float]] = []
        with open(labels_path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) != 5:
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: invalid values count: {line}")
                    continue
                try:
                    c, xc, yc, w, h = map(float, parts)
                except ValueError:
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: cannot parse: {line}")
                    continue

                if not (0.0 <= xc <= 1.0 and 0.0 <= yc <= 1.0 and 0.0 < w <= 1.0 and 0.0 < h <= 1.0):
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: invalid bbox: {line}")
                    continue

                labels.append([c, xc, yc, w, h])

        if len(labels) == 0:
            self.miss_labeled.append(labels_path)
            return torch.tensor([[-1, -1, -1, -1, -1]], dtype=torch.float32)

        return torch.tensor(labels, dtype=torch.float32)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        image_path = self.images[idx]
        img = decode_image(image_path)  # uint8 [C,H,W] on CPU
        if self.transforms is not None:
            img = self.transforms(img)

        labels_path = self._labels_path_from_image(image_path)
        targets = self._read_labels(labels_path)
        return img, targets

    def __len__(self) -> int:
        return len(self.images)


# -----------------------------
# Collate & Batch builder
# -----------------------------
def collate_fn(batch: list[tuple[Tensor, Tensor]]) -> tuple[Tensor, Tensor]:
    images, targets = zip(*batch)
    images = torch.stack(images, dim=0)
    padded_targets = pad_sequence(list(targets), batch_first=True, padding_value=-1.0)
    return images, padded_targets


def prepare_batch_dict(images: Tensor, targets: Tensor) -> dict:
    valid_mask = targets[..., 0] != -1
    if not valid_mask.any():
        return {
            "img": images,
            "batch_idx": torch.empty((0,), device=images.device, dtype=torch.int64),
            "cls": torch.empty((0,), device=images.device, dtype=torch.float32),
            "bboxes": torch.empty((0, 4), device=images.device, dtype=torch.float32),
        }

    valid_targets = targets[valid_mask]  # [M,5]

    per_img_counts = valid_mask.sum(dim=1).to(torch.int64)
    batch_idx_list = []
    for i, cnt in enumerate(per_img_counts.tolist()):
        if cnt > 0:
            batch_idx_list.append(torch.full((cnt,), i, device=images.device, dtype=torch.int64))
    batch_idx = (
        torch.cat(batch_idx_list, dim=0)
        if batch_idx_list
        else torch.empty((0,), device=images.device, dtype=torch.int64)
    )

    cls = valid_targets[:, 0].to(torch.float32)
    bboxes = valid_targets[:, 1:].to(torch.float32)
    return {"img": images, "batch_idx": batch_idx, "cls": cls, "bboxes": bboxes}


# -----------------------------
# Train / Valid
# -----------------------------
def train_one_epoch(
    model: YOLO,
    optimizer: Adam,
    dataloader: DataLoader,
    device: torch.device,
    epoch: int,
    max_train_steps: int = 0,
    debug_print_first_batch: bool = True,
) -> pd.DataFrame:
    model.model.train(True)
    torch.set_grad_enabled(True)
    loss_fn = v8DetectionLoss(model.model)

    rows = []
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"train epoch {epoch}")

    for step, (images, targets) in pbar:
        if max_train_steps > 0 and step >= max_train_steps:
            break

        images = images.to(device, non_blocking=True).float() / 255.0
        targets = targets.to(device, non_blocking=True)

        if debug_print_first_batch and step == 0:
            print("[DEBUG] images device:", images.device)
            print("[DEBUG] targets device:", targets.device)
            print("[DEBUG] model device:", next(model.model.parameters()).device)

        batch = prepare_batch_dict(images, targets)
        if batch["bboxes"].numel() == 0:
            pbar.set_postfix_str("skip(empty)")
            continue

        optimizer.zero_grad(set_to_none=True)

        # ✅ Prevent inference_mode leakage from Ultralytics internals in notebooks
        with torch.inference_mode(False):
            with torch.enable_grad():
                preds = model.model(images)
                batch_loss, last_loss = loss_fn(preds, batch)
                loss_scalar = batch_loss.sum()

        if not torch.isfinite(loss_scalar):
            pbar.set_postfix_str("skip(non-finite)")
            continue

        if not loss_scalar.requires_grad:
            raise RuntimeError(
                f"loss_scalar.requires_grad=False at step {step}. "
                f"Grad is disabled globally or inference_mode is active."
            )

        loss_scalar.backward()
        optimizer.step()

        box_loss, cls_loss, dfl_loss = last_loss.detach().cpu().tolist()
        row = {
            "epoch": epoch,
            "step": step,
            "box": float(box_loss),
            "cls": float(cls_loss),
            "dfl": float(dfl_loss),
        }
        rows.append(row)
        pbar.set_postfix({"box": f"{row['box']:.4f}", "cls": f"{row['cls']:.4f}", "dfl": f"{row['dfl']:.4f}"})

    return pd.DataFrame(rows)


@torch.no_grad()
def valid_loss_only(
    model: YOLO,
    dataloader: DataLoader,
    device: torch.device,
    epoch: int,
    max_val_steps: int = 0,
) -> pd.DataFrame:
    model.model.eval()
    loss_fn = v8DetectionLoss(model.model)

    rows = []
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"valid(loss) epoch {epoch}")

    for step, (images, targets) in pbar:
        if max_val_steps > 0 and step >= max_val_steps:
            break

        images = images.to(device, non_blocking=True).float() / 255.0
        targets = targets.to(device, non_blocking=True)

        batch = prepare_batch_dict(images, targets)
        if batch["bboxes"].numel() == 0:
            continue

        preds = model.model(images)
        _, last_loss = loss_fn(preds, batch)

        box_loss, cls_loss, dfl_loss = last_loss.detach().cpu().tolist()
        rows.append(
            {"epoch": epoch, "batch": step, "box": float(box_loss), "cls": float(cls_loss), "dfl": float(dfl_loss)}
        )

    return pd.DataFrame(rows)


def yolo_val_metrics(model: YOLO, data_yaml: str, device_id=0, workers: int = 0):
    # plots=False để tránh warning matplotlib & nhẹ hơn
    return model.val(data=data_yaml, device=device_id, workers=workers, verbose=False, plots=False)


# -----------------------------
# Main
# -----------------------------
@dataclass
class Config:
    base_path: str
    data_yaml: str
    epochs: int = 2
    batch_size: int = 4
    lr: float = 3e-4
    num_workers: int = 2
    img_ext: str = "jpg"
    label_ext: str = "txt"
    checkpoint_dir: str = "checkpoints"

    # Debug helpers (0 means "no limit")
    max_train_steps: int = 0
    max_val_steps: int = 0


def main(cfg: Config) -> None:
    device = get_device()
    print(f"[INFO] device = {device}")
    if device.type == "cuda":
        print("[INFO] CUDA available:", torch.cuda.is_available())
        print("[INFO] CUDA device:", torch.cuda.get_device_name(0))

    ensure_dir(cfg.checkpoint_dir)

    # model = YOLO(model="custom_yolov11n.yaml", task="detect", verbose=True)
    # model.model.args = SimpleNamespace(box=15.0, cls=0.5, dfl=2.25)  # type: ignore
    # model.model.to(device)  # type: ignore

    ####################################################################################################################################
    ######################################################## Swap Layer ################################################################
    # Swap CoordConv vào P2/4 và P3/8, bỏ swap HoangTN_CBAM
    ####################################################################################################################################

    # 1) load YOLO wrapper
    model = YOLO(model="custom_yolov11n.yaml", task="detect", verbose=True)

    # 2) set loss weights
    model.model.args = SimpleNamespace(box=15.0, cls=0.5, dfl=2.25)  # type: ignore

    # 3) SWAP: Conv stride=2 (P2/4 out=128) và (P3/8 out=256) -> CoordConv
    layers = model.model.model  # nn.ModuleList các layer

    def is_ultra_conv(m: nn.Module) -> bool:
        # Ultralytics Conv wrapper thường có attribute .conv (nn.Conv2d)
        return hasattr(m, "conv") and isinstance(getattr(m, "conv", None), nn.Conv2d)

    def get_conv_meta(m: nn.Module):
        # Trả về (in_ch, out_ch, stride) nếu lấy được, else None
        if is_ultra_conv(m):
            conv = m.conv
            # stride có thể là tuple (sH,sW)
            s = conv.stride[0] if isinstance(conv.stride, tuple) else int(conv.stride)
            return conv.in_channels, conv.out_channels, s
        return None

    targets = {"p2": None, "p3": None}  # store indices

    # Tìm Conv stride=2 có out=128 (P2/4) và out=256 (P3/8)
    for i, m in enumerate(layers):
        meta = get_conv_meta(m)
        if meta is None:
            continue
        c1, c2, s = meta
        if s == 2 and c2 == 128 and targets["p2"] is None:
            targets["p2"] = i
        if s == 2 and c2 == 256 and targets["p3"] is None:
            targets["p3"] = i

    # Nếu không tìm thấy, in debug tất cả Conv stride=2 để bạn đối chiếu
    if targets["p2"] is None or targets["p3"] is None:
        print("\n[DEBUG] List Conv(stride=2):")
        for i, m in enumerate(layers):
            meta = get_conv_meta(m)
            if meta is None:
                continue
            c1, c2, s = meta
            if s == 2:
                print(f"  idx={i:3d}  Conv  {c1}->{c2}  stride={s}")
        raise RuntimeError(f"Không tìm thấy Conv stride=2 cho P2/4(out=128) hoặc P3/8(out=256). Found={targets}")

    # Swap P2/4
    idx_p2 = targets["p2"]
    old_p2 = layers[idx_p2]
    c1_p2, c2_p2, _s_p2 = get_conv_meta(old_p2)
    new_p2 = CoordConv(c1_p2, c2_p2, k=3, s=2, normalize=True)

    # Copy Ultralytics metadata
    for k in ("f", "i", "type", "np"):
        if hasattr(old_p2, k):
            setattr(new_p2, k, getattr(old_p2, k))
    new_p2.training = old_p2.training
    layers[idx_p2] = new_p2
    print(f"[SWAP OK] P2/4 at idx={idx_p2}: Conv({c1_p2}->{c2_p2}, s=2) -> CoordConv")

    # Swap P3/8
    idx_p3 = targets["p3"]
    old_p3 = layers[idx_p3]
    c1_p3, c2_p3, _s_p3 = get_conv_meta(old_p3)
    new_p3 = CoordConv(c1_p3, c2_p3, k=3, s=2, normalize=True)

    for k in ("f", "i", "type", "np"):
        if hasattr(old_p3, k):
            setattr(new_p3, k, getattr(old_p3, k))
    new_p3.training = old_p3.training
    layers[idx_p3] = new_p3
    print(f"[SWAP OK] P3/8 at idx={idx_p3}: Conv({c1_p3}->{c2_p3}, s=2) -> CoordConv")

    # 4) move to device
    model.model.to(device)  # type: ignore
    ####################################################################################################################################
    ####################################################################################################################################
    print("\n================ MODEL INFO (after swap) ================\n", flush=True)
    model.model.info()

    print("\n================ LAYERS (after swap) ================\n", flush=True)

    swapped_idxs = {idx_p2, idx_p3}

    for i, m in enumerate(model.model.model):
        name = m.__class__.__name__
        f = getattr(m, "f", None)

        np_ = sum(p.numel() for p in m.parameters()) if isinstance(m, nn.Module) else 0

        c1 = c2 = None
        if hasattr(m, "cv1") and hasattr(m, "cv2") and hasattr(m.cv1, "conv") and hasattr(m.cv2, "conv"):
            try:
                c1 = m.cv1.conv.in_channels
                c2 = m.cv2.conv.out_channels
            except Exception:
                pass

        mark = "  <== SWAPPED" if i in swapped_idxs else ""

        if c1 is not None and c2 is not None:
            print(f"{i:3d}  f={f!s:>6}  {name:<20}  c1->c2: {c1:4d}->{c2:<4d}  params={np_:>8d}{mark}")
        else:
            print(f"{i:3d}  f={f!s:>6}  {name:<20}  params={np_:>8d}{mark}")

    ####################################################################################################################################

    print("[DEBUG] model param device:", next(model.model.parameters()).device)

    optimizer = Adam(model.model.parameters(), lr=cfg.lr)

    train_dataset = AodDataset(
        images_dir=os.path.join(cfg.base_path, "train/images"),
        labels_dir=os.path.join(cfg.base_path, "train/labels"),
        images_ext=cfg.img_ext,
        labels_ext=cfg.label_ext,
        verbose_bad_labels=False,
    )
    val_dataset = AodDataset(
        images_dir=os.path.join(cfg.base_path, "valid/images"),
        labels_dir=os.path.join(cfg.base_path, "valid/labels"),
        images_ext=cfg.img_ext,
        labels_ext=cfg.label_ext,
        verbose_bad_labels=False,
    )

    pin = torch.cuda.is_available()
    workers = cfg.num_workers

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=workers,
        pin_memory=pin,
        persistent_workers=(workers > 0),
        drop_last=False,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=workers,
        pin_memory=pin,
        persistent_workers=(workers > 0),
        drop_last=False,
    )

    train_all = []

    best_score = float("inf")
    best_epoch = -1
    best_path = os.path.join(cfg.checkpoint_dir, "best.pt")
    last_path = os.path.join(cfg.checkpoint_dir, "last.pt")

    for epoch in range(1, cfg.epochs + 1):
        print(f"\n==================== EPOCH {epoch}/{cfg.epochs} ====================")

        train_df = train_one_epoch(
            model,
            optimizer,
            train_loader,
            device,
            epoch,
            max_train_steps=cfg.max_train_steps,
            debug_print_first_batch=(epoch == 1),
        )
        train_all.append(train_df)

        if len(train_df) > 0:
            train_score = train_df[["box", "cls", "dfl"]].mean().sum()
            print(f"[EPOCH {epoch}] train_score={train_score:.6f} | best_score={best_score:.6f}")

            if train_score < best_score:
                best_score = train_score
                best_epoch = epoch
                model.save(best_path)
                print(f"[BEST] updated → epoch {best_epoch}, saved to {best_path}")
        else:
            print("[TRAIN] no valid batches produced loss (all empty labels?)")

        model.save(last_path)
        ckpt_path = os.path.join(cfg.checkpoint_dir, f"epoch_{epoch}.pt")
        model.save(ckpt_path)
        print(f"[INFO] saved checkpoint: {ckpt_path} (and last.pt)")

    train_csv = os.path.join(cfg.checkpoint_dir, "train_losses.csv")
    pd.concat(train_all, axis=0, ignore_index=True).to_csv(train_csv, index=False)
    print(f"[INFO] saved train logs: {train_csv}")

    print("\n==================== TRAIN SUMMARY ====================")
    print(f"[BEST MODEL] epoch = {best_epoch}")
    print(f"[BEST MODEL] score = {best_score:.6f}")
    print(f"[BEST MODEL] path  = {best_path}")
    print(f"[LAST MODEL] path  = {last_path}")

    print("\n==================== FINAL EVALUATION ====================")

    eval_ckpt = best_path if os.path.exists(best_path) else last_path
    print(f"[EVAL] loading checkpoint: {eval_ckpt}")

    eval_model = YOLO(eval_ckpt)
    eval_model.model.args = SimpleNamespace(box=15.0, cls=0.5, dfl=2.25)  # type: ignore
    eval_model.model.to(device)  # type: ignore

    val_loss_df = valid_loss_only(
        eval_model,
        val_loader,
        device,
        epoch=best_epoch if os.path.exists(best_path) else cfg.epochs,
        max_val_steps=cfg.max_val_steps,
    )
    val_csv = os.path.join(cfg.checkpoint_dir, "val_losses.csv")
    val_loss_df.to_csv(val_csv, index=False)
    if len(val_loss_df) > 0:
        print("[FINAL VALID-LOSS] mean:", val_loss_df[["box", "cls", "dfl"]].mean().to_dict())
    else:
        print("[FINAL VALID-LOSS] no valid batches (empty labels?)")
    print(f"[INFO] saved val losses: {val_csv}")

    try:
        print("[FINAL VALID-METRICS] running ultralytics val() ...")
        device_id = 0 if device.type == "cuda" else "cpu"
        metrics = yolo_val_metrics(eval_model, cfg.data_yaml, device_id=device_id, workers=cfg.num_workers)

        print(f"Precision: {metrics.box.mp:.4f}")
        print(f"Recall:    {metrics.box.mr:.4f}")
        print(f"mAP@0.5:   {metrics.box.map50:.4f}")
        print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
    except Exception as e:
        print(f"[WARN] final model.val() failed: {e}")


if __name__ == "__main__":

    def running_in_notebook():
        try:
            from IPython import get_ipython

            return get_ipython() is not None
        except Exception:
            return False

    if running_in_notebook():
        print("[INFO] Running in Kaggle/Colab Notebook → using hardcoded config")
        cfg = Config(
            base_path="/kaggle/input/aod-4-yolo",
            data_yaml="/kaggle/working/data.yaml",
            epochs=4,
            batch_size=4,
            lr=3e-4,
            num_workers=2,  # ✅ try 2; set 0 if you see worker issues
            max_train_steps=1000,
            max_val_steps=0,
        )
        main(cfg)
    else:
        parser = argparse.ArgumentParser()
        parser.add_argument("--base_path", type=str, required=True)
        parser.add_argument("--data_yaml", type=str, required=True)
        parser.add_argument("--epochs", type=int, default=2)
        parser.add_argument("--batch_size", type=int, default=4)
        parser.add_argument("--lr", type=float, default=3e-4)
        parser.add_argument("--num_workers", type=int, default=2)
        parser.add_argument("--img_ext", type=str, default="jpg")
        parser.add_argument("--label_ext", type=str, default="txt")
        parser.add_argument("--checkpoint_dir", type=str, default="checkpoints")
        parser.add_argument("--max_train_steps", type=int, default=0)
        parser.add_argument("--max_val_steps", type=int, default=0)

        args = parser.parse_args()
        cfg = Config(**vars(args))
        main(cfg)

[INFO] Running in Kaggle/Colab Notebook → using hardcoded config
[INFO] device = cuda:0
[INFO] CUDA available: True
[INFO] CUDA device: Tesla P100-PCIE-16GB

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultral

train epoch 1:   0%|          | 0/3941 [00:00<?, ?it/s]

[DEBUG] images device: cuda:0
[DEBUG] targets device: cuda:0
[DEBUG] model device: cuda:0
[EPOCH 1] train_score=15.076489 | best_score=inf
[BEST] updated → epoch 1, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_1.pt (and last.pt)

==================== EPOCH 2/4 ====================


train epoch 2:   0%|          | 0/3941 [00:00<?, ?it/s]

[EPOCH 2] train_score=11.232085 | best_score=15.076489
[BEST] updated → epoch 2, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_2.pt (and last.pt)

==================== EPOCH 3/4 ====================


...

[EPOCH 3] train_score=10.179157 | best_score=11.232085
[BEST] updated → epoch 3, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_3.pt (and last.pt)

==================== EPOCH 4/4 ====================


train epoch 4:   0%|          | 0/3941 [00:00<?, ?it/s]

[EPOCH 4] train_score=9.615683 | best_score=10.179157
[BEST] updated → epoch 4, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_4.pt (and last.pt)
[INFO] saved train logs: checkpoints/train_losses.csv

==================== TRAIN SUMMARY ====================
[BEST MODEL] epoch = 4
[BEST MODEL] score = 9.615683
[BEST MODEL] path  = checkpoints/best.pt
[LAST MODEL] path  = checkpoints/last.pt

==================== FINAL EVALUATION ====================
[EVAL] loading checkpoint: checkpoints/best.pt


valid(loss) epoch 4:   0%|          | 0/1129 [00:00<?, ?it/s]

[FINAL VALID-LOSS] mean: {'box': 4.09735773080133, 'cls': 2.431747179181052, 'dfl': 3.5157533848766787}
[INFO] saved val losses: checkpoints/val_losses.csv
[FINAL VALID-METRICS] running ultralytics val() ...
Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
custom_YOLOv11n summary (fused): 101 layers, 2,589,844 parameters, 13,260 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 88.0±36.8 MB/s, size: 47.8 KB)
val: Scanning /kaggle/input/aod-4-yolo/valid/labels... 4514 images, 125 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4514/4514 1.4Kit/s 3.3s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/aod-4-yolo/valid is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 283/283 11.8it/s 24.0s0.2ss
                   all       4514       6369      0.276      0.274      0.199     0.0773
Speed: 0.7ms preprocess, 2.2ms inference, 0.0ms loss, 0.8m